I am using the Dataset DSTC8 SGD to develop the Chatbot

In [ ]:
!git clone https://github.com/google-research-datasets/dstc8-schema-guided-dialogue.git


Cloning into 'dstc8-schema-guided-dialogue'...
remote: Enumerating objects: 711, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 711 (delta 12), reused 8 (delta 8), pack-reused 697 (from 1)
Receiving objects: 100% (711/711), 49.89 MiB | 10.81 MiB/s, done.
Resolving deltas: 100% (593/593), done.
Updating files: 100% (208/208), done.


I am extracting the utterance-intent pairs from the dataset

In [ ]:
import json
import os

def extract_user_intents(path):
    data = []
    for file in os.listdir(path):
        if file.startswith("dialogues_") and file.endswith(".json"):
            with open(os.path.join(path, file), "r") as f:
                dialogs = json.load(f)
                for dialog in dialogs:
                    if "turns" not in dialog:
                        continue  # Skip malformed entries
                    for turn in dialog["turns"]:
                        if turn.get("speaker") == "USER":
                            utterance = turn.get("utterance", "")
                            for frame in turn.get("frames", []):
                                intent = frame.get("state", {}).get("active_intent", "NONE")
                                if intent != "NONE":
                                    data.append((utterance, intent))
    return data

# ✅ Run the fixed version
train_data = extract_user_intents("/content/dstc8-schema-guided-dialogue/train")
print(f"Extracted {len(train_data)} utterance-intent pairs")
train_data[:5]


Extracted 163196 utterance-intent pairs


[("I'd like to get some tickets for an event.", 'BuyEventTickets'),
 ("I'm looking for events next Wednesday in the Stanford area. I'd need 3 tickets.",
  'BuyEventTickets'),
 ("Yes, I'm intersted in Nycfc Vs Dynamo. It's actually near NY.",
  'BuyEventTickets'),
 ('That is correct.', 'BuyEventTickets'),
 ("That's too bad. Can you check to see if I can get tickets to the Giants Vs Dodgers?",
  'BuyEventTickets')]

✅ Step 1: Convert to DataFrame & Explore

In [ ]:
import pandas as pd

# Convert to DataFrame
df = pd.DataFrame(train_data, columns=["text", "label"])

# Remove duplicates and "NONE" labels (if any remain)
df = df.drop_duplicates()
df = df[df["label"] != "NONE"]
df = df.reset_index(drop=True)

# Inspect
print(f"Total samples: {len(df)}")
df["label"].value_counts().head(10)


Total samples: 136389


,count
label,
BuyEventTickets,7801
ReserveRestaurant,7366
FindEvents,7110
BookAppointment,6852
FindBus,6688
FindRestaurants,6284
ReserveHotel,6242
SearchRoundtripFlights,6114
GetRide,6056


In [ ]:
df.to_csv("dstc8_intent_dataset.csv", index=False)

from google.colab import files
files.download("dstc8_intent_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Now I'm going to train a BERT-based intent classifier using DSTC8 data.

🔧 Step 0: Install Hugging Face Libraries

In [ ]:
!pip install transformers datasets scikit-learn


📊 Step 1: Prepare Dataset (Tokenization + Encoding Labels)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset

# Encode labels into integers
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])

# Save label encoder mappings
id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in enumerate(label_encoder.classes_)}

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Convert pandas DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df[["text", "label_id"]])

# Tokenize the text
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

dataset = dataset.map(tokenize, batched=True)

# Load DistilBERT model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_encoder.classes_),
    id2label=id2label,
    label2id=label2id
)


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/136389 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertForSequenceClassification: ['vocab_transform.bias', 'vocab_projector.bias', 'vocab_layer_norm.weight', 'vocab_transform.weight', 'vocab_layer_norm.bias']
- This IS expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification mod

In [ ]:
🧪 Step 2: Train-Test Split

In [ ]:
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [ ]:
train_dataset = train_dataset.rename_columns({"label_id": "labels"})
test_dataset = test_dataset.rename_columns({"label_id": "labels"})

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Combine all intents from train and test data
all_intents = [label for _, label in train_data]


# Fit LabelEncoder to convert intent strings to numeric IDs
label_encoder = LabelEncoder()
label_encoder.fit(all_intents)

# Create mappings
id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in enumerate(label_encoder.classes_)}

🧠 Step 3: Load BERT Model

⚙️ Step 4: Set Training Arguments

In [ ]:
!pip install transformers==4.28.1

📈 Step 5: Define Evaluation Metrics

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {"accuracy": acc, "f1": f1}


🏋️ Step 6: Train the Model

In [ ]:
# 4. Define training arguments
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,  # mixed precision for speed
    gradient_accumulation_steps=2
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:645: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


Training the model

In [ ]:
trainer.train()


/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vavilalalasyapriya (vavilalalasyapriya-arizona-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:2664: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  else torch.cuda.amp.autocast(cache_enabled=cache_enabled, dtype=self.amp_dtype)


Epoch,Training Loss,Validation Loss,Accuracy,F1
0,1.244900,1.188233,0.626512,0.621704
2,0.813900,1.130428,0.638243,0.634708


/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:2664: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  else torch.cuda.amp.autocast(cache_enabled=cache_enabled, dtype=self.amp_dtype)
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:2664: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  else torch.cuda.amp.autocast(cache_enabled=cache_enabled, dtype=self.amp_dtype)


Epoch,Training Loss,Validation Loss,Accuracy,F1
0,1.244900,1.188233,0.626512,0.621704
2,0.813900,1.130428,0.638243,0.634708
2,0.833100,1.128337,0.637510,0.633598


TrainOutput(global_step=92061, training_loss=1.1672288172728875, metrics={'train_runtime': 10554.7132, 'train_samples_per_second': 34.89, 'train_steps_per_second': 8.722, 'total_flos': 4.880990229730099e+16, 'train_loss': 1.1672288172728875, 'epoch': 3.0})

In [ ]:
import torch

def predict_intent(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)
    outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=1).item()
    return id2label[prediction]

# Example
predict_intent("I'd like to reset my password")

np.str_('ReserveHotel')

In [ ]:
# Save model and tokenizer
model.save_pretrained("./intent_model")
tokenizer.save_pretrained("./intent_model")

# Later you can reload
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("./intent_model")
tokenizer = AutoTokenizer.from_pretrained("./intent_model")


In [ ]:
from huggingface_hub import notebook_login

notebook_login()


In [ ]:
model.push_to_hub("lasyapriyav/intent-classifier-bert")
tokenizer.push_to_hub("lasyapriyav/intent-classifier-bert")


Uploading...:   0%|          | 0.00/268M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/lasyapriyav/intent-classifier-bert/commit/d4f948947507920f672736989aa80304cb1d0bb8', commit_message='Upload tokenizer', commit_description='', oid='d4f948947507920f672736989aa80304cb1d0bb8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/lasyapriyav/intent-classifier-bert', endpoint='https://huggingface.co', repo_type='model', repo_id='lasyapriyav/intent-classifier-bert'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "lasyapriyav/intent-classifier-bert"

# Download and save locally
model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Save to local directory
save_directory = "./intent_classifier_model"
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [ ]:
pip install gradio


In [ ]:
import gradio as gr
from transformers import pipeline

# Load model from Hugging Face Hub (your model repo)
model_name = "lasyapriyav/intent-classifier-bert"
classifier = pipeline("text-classification", model=model_name)

# Inference function
def predict_intent(text):
    result = classifier(text)[0]
    return f"Intent: {result['label']} (Confidence: {round(result['score'], 2)})"

# Launch Gradio interface
gr.Interface(
    fn=predict_intent,
    inputs=gr.Textbox(lines=2, placeholder="Enter a support query..."),
    outputs="text",
    title="Intent Classifier Chatbot",
    description="This chatbot predicts the intent of user queries using a fine-tuned DistilBERT model."
).launch()


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cpu


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f84eda9044bce99154.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
